In [1]:
import re
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from collections import Counter
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

In [21]:
categories = ["sci.space", "rec.sport.baseball"]

data = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes")
)

texts = data.data
labels = data.target

print(len(texts))
print(categories)
print(texts[0][:300])

1981
['sci.space', 'rec.sport.baseball']

Do you really have *that* much information on him?  Really?


I don't know.  You tell me.  What percentage of players reach or 
exceed their MLE's *in their rookie season*?  We're talking about
1993, you know.


If that were your purpose, maybe.  Offerman spent 1992 getting 
acclimated, if you will


In [22]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

tokens = tokenize("BERT learns contextual word representations")
print(tokens)

['bert', 'learns', 'contextual', 'word', 'representations']


In [23]:
PAD = "[PAD]"
UNK = "[UNK]"
CLS = "[CLS]"
SEP = "[SEP]"

counter = Counter()

for text in texts:
    counter.update(tokenize(text))

vocab_size = 8000

vocab = {
    PAD: 0,
    UNK: 1,
    CLS: 2,
    SEP: 3
}

for word, freq in counter.most_common(vocab_size - 4):
    vocab[word] = len(vocab)

print("Vocab size:", len(vocab))

Vocab size: 8000


In [45]:
max_len = 128

def encode_text(text):
    tokens = [CLS] + tokenize(text)[:max_len - 2] + [SEP]

    ids = [vocab.get(tok, vocab[UNK]) for tok in tokens]

    attention_mask = [1] * len(ids)

    while len(ids) < max_len:
        ids.append(vocab[PAD])
        attention_mask.append(0)

    return torch.tensor(ids), torch.tensor(attention_mask)

In [46]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
       
        return len(self.texts)

    def __getitem__(self, idx):
        input_ids, attention_mask = encode_text(self.texts[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return input_ids, attention_mask, label

In [47]:
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

train_dataset = NewsDataset(X_train, y_train)
test_dataset = NewsDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

BERT embedding layer

In [48]:
class BertEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_len):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size, embed_dim)
        self.position_embeddings = nn.Embedding(max_len, embed_dim)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)

        token_emb = self.token_embeddings(input_ids)
        pos_emb = self.position_embeddings(positions)
        return token_emb + pos_emb

multi head attention 

In [49]:
class multiheadselfattention(nn.Module):
       def __init__(self,embed_dim,num_heads):
              super().__init__()
              assert embed_dim % num_heads==0 
              self.embed_dim = embed_dim
              self.num_heads = num_heads
              self.head_dim = embed_dim // num_heads
              self.q_linear = nn.Linear(embed_dim, embed_dim)
              self.k_linear=nn.Linear(embed_dim, embed_dim)
              self.v_linear=nn.Linear(embed_dim, embed_dim)
              self.out=nn.Linear(embed_dim, self.embed_dim)

       def forward(self,x,attention_mask):
              B,S,E=x.shape
              Q=self.q_linear(x).view(B,S,self.num_heads,self.head_dim).transpose(1,2)
              K=self.k_linear(x).view(B,S,self.num_heads,self.head_dim).transpose(2,1)
              V=self.v_linear(x).view(B,S,self.num_heads,self.head_dim).transpose(1,2)
              scores=torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(self.head_dim)
              scores=scores.masked_fill(attention_mask.unsqueeze(1).unsqueeze(2)==0,float("-inf"))
              mask=attention_mask.unsqueeze(1).unsqueeze(2)
              scores=scores.masked_fill(mask==0,-1e9)

              attention_weights=F.softmax(scores,dim=-1)
              context=torch.matmul(attention_weights,V)
              context=context.transpose(1,2).contiguous().view(B,S,E)
              context=context.view(B,S,E)
              return self.out(context)

feef forwad network 

In [50]:
class feedforward(nn.Module):
       def __init__(self,embed_dim,hidden_dim):
              super().__init__()
              self.fc1=nn.Linear(embed_dim,hidden_dim)
              self.fc2=nn.Linear(hidden_dim,embed_dim)

       def forward(self,x):
              x=self.fc1(x)
              x=F.gelu(x)
              x=self.fc2(x)
              return x

transformer encoder block

In [51]:
class TransformerEncoderBlock(nn.Module):
       def __init__(self,embed_dim,num_heads,ff_hidden_dim,dropout=0.1):
              super().__init__()

              self.attention=multiheadselfattention(embed_dim,num_heads)
              self.norm1=nn.LayerNorm(embed_dim)
              self.ff=feedforward(embed_dim,ff_hidden_dim)
              self.norm2=nn.LayerNorm(embed_dim)
              self.dropout=nn.Dropout(dropout)
       def forward(self,x,attention_mask):
              attn_output=self.attention(x,attention_mask)
              x=self.norm1(x+self.dropout(attn_output))
              ff_output=self.ff(x)
              x=self.norm2(x+self.dropout(ff_output))
              return x

full bert model

In [52]:
# =========================
# 12. Mini-BERT Classifier
# =========================
class MiniBERTClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        max_len,
        num_classes,
        embed_dim=128,
        num_heads=4,
        ff_hidden_dim=256,
        num_layers=2,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = BertEmbedding(vocab_size, embed_dim, max_len)

        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(
                embed_dim,
                num_heads,
                ff_hidden_dim,
                dropout
            )
            for _ in range(num_layers)
        ])

        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids)

        for layer in self.encoder_layers:
            x = layer(x, attention_mask)

        cls_output = x[:, 0, :]

        logits = self.classifier(cls_output)

        return logits

In [53]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniBERTClassifier(
    vocab_size=len(vocab),
    max_len=max_len,
    num_classes=2
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
criterion = nn.CrossEntropyLoss()

print(model)

MiniBERTClassifier(
  (embedding): BertEmbedding(
    (token_embeddings): Embedding(8000, 128)
    (position_embeddings): Embedding(128, 128)
  )
  (encoder_layers): ModuleList(
    (0-1): 2 x TransformerEncoderBlock(
      (attention): multiheadselfattention(
        (q_linear): Linear(in_features=128, out_features=128, bias=True)
        (k_linear): Linear(in_features=128, out_features=128, bias=True)
        (v_linear): Linear(in_features=128, out_features=128, bias=True)
        (out): Linear(in_features=128, out_features=128, bias=True)
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ff): feedforward(
        (fc1): Linear(in_features=128, out_features=256, bias=True)
        (fc2): Linear(in_features=256, out_features=128, bias=True)
      )
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)


In [55]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for input_ids, attention_mask, labels_batch in train_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)

        loss = criterion(logits, labels_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels_batch).sum().item()
        total += labels_batch.size(0)

    train_acc = correct / total

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Loss: {total_loss / len(train_loader):.4f}")
    print(f"Train Accuracy: {train_acc:.4f}")

Epoch 1/10
Loss: 0.0340
Train Accuracy: 0.9804
Epoch 2/10
Loss: 0.0343
Train Accuracy: 0.9766
Epoch 3/10
Loss: 0.0328
Train Accuracy: 0.9779
Epoch 4/10
Loss: 0.0313
Train Accuracy: 0.9811
Epoch 5/10
Loss: 0.0324
Train Accuracy: 0.9785
Epoch 6/10
Loss: 0.0314
Train Accuracy: 0.9830
Epoch 7/10
Loss: 0.0307
Train Accuracy: 0.9817
Epoch 8/10
Loss: 0.0307
Train Accuracy: 0.9792
Epoch 9/10
Loss: 0.0307
Train Accuracy: 0.9792
Epoch 10/10
Loss: 0.0303
Train Accuracy: 0.9817


In [56]:

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for input_ids, attention_mask, labels_batch in test_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels_batch = labels_batch.to(device)

        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)

        correct += (preds == labels_batch).sum().item()
        total += labels_batch.size(0)

test_acc = correct / total

print("Test Accuracy:", test_acc)

Test Accuracy: 0.836272040302267


In [60]:
def evaluate_model(loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for input_ids, attention_mask, labels_batch in loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels_batch = labels_batch.to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels_batch)
            preds = torch.argmax(logits, dim=1)

            total_loss += loss.item()
            correct += (preds == labels_batch).sum().item()
            total += labels_batch.size(0)

    return total_loss / len(loader), correct / total


train_loss, train_acc = evaluate_model(train_loader)
test_loss, test_acc = evaluate_model(test_loader)

acc_gap = train_acc - test_acc
loss_gap = test_loss - train_loss

print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc:.4f}")
print(f"Test  Loss: {test_loss:.4f} | Test  Accuracy: {test_acc:.4f}")
print(f"Accuracy Gap: {acc_gap:.4f}")
print(f"Loss Gap: {loss_gap:.4f}")

if train_acc > 0.90 and acc_gap > 0.10:
    print("Result: Model overfit kortese. Train accuracy onek beshi, test accuracy kom.")
elif train_acc < 0.70 and test_acc < 0.70:
    print("Result: Model underfit kortese. Train/test duita accuracy-i kom.")
else:
    print("Result: Strong overfitting dekha jacche na. Train-test gap acceptable.")

Train Loss: 0.0315 | Train Accuracy: 0.9817
Test  Loss: 0.8801 | Test  Accuracy: 0.8363
Accuracy Gap: 0.1454
Loss Gap: 0.8486
Result: Model overfit kortese. Train accuracy onek beshi, test accuracy kom.


In [62]:
# Anti-overfitting retrain cell
# Goal: overfit vanish na, reduce kora. Regularization + validation + early stopping.

train_texts, val_texts, train_labels, val_labels = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

train_loader = DataLoader(NewsDataset(train_texts, train_labels), batch_size=32, shuffle=True)
val_loader = DataLoader(NewsDataset(val_texts, val_labels), batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

model = MiniBERTClassifier(
    vocab_size=len(vocab),
    max_len=max_len,
    num_classes=len(data.target_names),
    embed_dim=64,
    num_heads=4,
    ff_hidden_dim=128,
    num_layers=1,
    dropout=0.4
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)


def run_epoch(loader, training=False):
    model.train() if training else model.eval()
    total_loss = 0
    correct = 0
    total = 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for input_ids, attention_mask, labels_batch in loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels_batch = labels_batch.to(device)

            if training:
                optimizer.zero_grad()

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels_batch)

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            preds = torch.argmax(logits, dim=1)
            total_loss += loss.item()
            correct += (preds == labels_batch).sum().item()
            total += labels_batch.size(0)

    return total_loss / len(loader), correct / total


best_val_loss = float("inf")
best_state = None
patience = 3
bad_epochs = 0
epochs = 15

for epoch in range(epochs):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= patience:
        print("Early stopping: validation loss improve kortese na.")
        break

if best_state is not None:
    model.load_state_dict(best_state)

train_loss, train_acc = run_epoch(train_loader)
val_loss, val_acc = run_epoch(val_loader)
test_loss, test_acc = run_epoch(test_loader)

print("\nAfter anti-overfit retraining:")
print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc:.4f}")
print(f"Val   Loss: {val_loss:.4f} | Val   Accuracy: {val_acc:.4f}")
print(f"Test  Loss: {test_loss:.4f} | Test  Accuracy: {test_acc:.4f}")
print(f"Train-Test Accuracy Gap: {train_acc - test_acc:.4f}")

if train_acc - test_acc > 0.10:
    print("Still overfit ache. More dropout/less epoch/aro data lagbe.")
else:
    print("Overfit onekta reduce hoyeche.")

Epoch 1/15 | train_loss=0.6981, train_acc=0.4933 | val_loss=0.6913, val_acc=0.5331
Epoch 2/15 | train_loss=0.6964, train_acc=0.5012 | val_loss=0.6888, val_acc=0.5457
Epoch 3/15 | train_loss=0.6882, train_acc=0.5399 | val_loss=0.6859, val_acc=0.5741
Epoch 4/15 | train_loss=0.6856, train_acc=0.5635 | val_loss=0.6832, val_acc=0.5931
Epoch 5/15 | train_loss=0.6845, train_acc=0.5509 | val_loss=0.6799, val_acc=0.5962
Epoch 6/15 | train_loss=0.6713, train_acc=0.6156 | val_loss=0.6751, val_acc=0.6088
Epoch 7/15 | train_loss=0.6706, train_acc=0.6030 | val_loss=0.6682, val_acc=0.6309
Epoch 8/15 | train_loss=0.6598, train_acc=0.6361 | val_loss=0.6532, val_acc=0.7035
Epoch 9/15 | train_loss=0.6352, train_acc=0.6709 | val_loss=0.6156, val_acc=0.7098
Epoch 10/15 | train_loss=0.5965, train_acc=0.6946 | val_loss=0.5790, val_acc=0.7256
Epoch 11/15 | train_loss=0.5706, train_acc=0.7024 | val_loss=0.5683, val_acc=0.7256
Epoch 12/15 | train_loss=0.5548, train_acc=0.7253 | val_loss=0.5598, val_acc=0.7256
E

In [63]:
def predict(text):
    model.eval()

    input_ids, attention_mask = encode_text(text)
    input_ids = input_ids.unsqueeze(0).to(device)
    attention_mask = attention_mask.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probabilities = torch.softmax(logits, dim=1).squeeze(0)
        predicted_class = torch.argmax(probabilities).item()

    class_names = data.target_names
    label = class_names[predicted_class]
    confidence = probabilities[predicted_class].item()

    print(f"Prediction: {label} ({confidence:.4f})")
    return label, confidence

In [67]:
# Stable baseline: TF-IDF + Logistic Regression
# Mini-BERT from scratch choto dataset-e unstable. Ei model overfit kom kore.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

tfidf_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    class_weight="balanced",
    random_state=42
)
tfidf_model.fit(X_train_tfidf, y_train)

train_probs = tfidf_model.predict_proba(X_train_tfidf)
test_probs = tfidf_model.predict_proba(X_test_tfidf)
train_pred = train_probs.argmax(axis=1)
test_pred = test_probs.argmax(axis=1)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)
train_loss = log_loss(y_train, train_probs)
test_loss = log_loss(y_test, test_probs)

print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc:.4f}")
print(f"Test  Loss: {test_loss:.4f} | Test  Accuracy: {test_acc:.4f}")
print(f"Accuracy Gap: {train_acc - test_acc:.4f}")


def predict(text, unknown_threshold=0.70, min_matched_features=2):
    features = tfidf.transform([text])

    if features.nnz < min_matched_features:
        print(f"Prediction: unknown/off-topic (matched_features={features.nnz})")
        return "unknown/off-topic", 0.0

    probabilities = tfidf_model.predict_proba(features)[0]
    predicted_class = probabilities.argmax()
    label = data.target_names[predicted_class]
    confidence = probabilities[predicted_class]

    if confidence < unknown_threshold:
        print(f"Prediction: unknown/off-topic ({confidence:.4f}); closest={label}")
        return "unknown/off-topic", confidence

    print(f"Prediction: {label} ({confidence:.4f})")
    return label, confidence

Train Loss: 0.3027 | Train Accuracy: 0.9729
Test  Loss: 0.3576 | Test  Accuracy: 0.9773
Accuracy Gap: -0.0045


In [69]:
predict("NASA launched a new spacecraft to explore Mars")

Prediction: sci.space (0.8426)


('sci.space', np.float64(0.8426475675973112))

In [70]:
predict("Hasina ekta mc")

Prediction: unknown/off-topic (matched_features=1)


('unknown/off-topic', 0.0)

In [71]:
predict("The baseball game was exciting and went into extra innings")

Prediction: rec.sport.baseball (0.8229)


('rec.sport.baseball', np.float64(0.8228510831019786))